# Multiyear decarbonisation of a chemical cluster (rolling horizon)

In this case study we optimize the transition of a chemical and petrochemical cluster in the Netherlands over three
investment intervals, each 10 years apart. The cluster has to satisfy a constant demand for ethylene, propylene,
methanol, ammonia and hydrogen. It can choose between conventional technologies (naphtha cracker, steam methane
reformer) and electrified or lower-emission alternatives (electric cracker, e-SMR, electrolyzer, electric boiler,
carbon capture with MEA).

The three intervals are solved one after the other (myopic foresight, rolling horizon). The capacities built in one
interval are carried over as existing capacity to the next one, as long as their lifetime has not run out:

- **Interval 1** (greenfield): the cluster is built from scratch, no emission limit.
- **Interval 2** (brownfield): net emissions are limited to 50% of the emissions of Interval 1.
- **Interval 3** (brownfield): net emissions are limited to zero.

The case study was set up by Alessandro Magnino. All data are provided with the package. It is a simplified version of a more extensive case study, with seven clusters in the Netherlands, Belgium and Germany, five investment years from 2020 to 2050, and hydrogen and CO2 networks. For this more complex development of the case study, see [Alessandro Magnino's fork of AdOpT-NET0](https://github.com/AlessandroMagnino/AdOpT-NET0_my/tree/ARRRA_paper).

## Settings
We define the folders for the input data and the results, the investment intervals and the number of years between
them. The years between intervals are used to check the remaining lifetime of the carried-over capacities and to
discount the cost of each interval in the postprocessing. The emission reduction of each interval is given relative
to the emissions of Interval 1.

The results are saved in their own folder. Since the postprocessing expects one row per interval in the summary file,
the input data and the results of a previous run of this case study are removed first.

In [ ]:
import json
import shutil
from pathlib import Path
import pandas as pd
import adopt_net0 as adopt
import adopt_net0.data_preprocessing as dp
from adopt_net0.modelhub import ModelHub
from adopt_net0.utilities import installed_capacities_existing
from adopt_net0.result_management.read_results import (
    add_values_to_summary,
    add_carry_over_annualization_to_summary,
    add_discounted_cost_to_summary,
)

# Folder for results and input data path (one case folder per interval is created in here)
results_data_path = Path("./userData/multiyear")
casestudy_path = Path("./caseStudies/multiyear")

# Remove the input data and results of a previous run of this case study
shutil.rmtree(results_data_path, ignore_errors=True)
shutil.rmtree(casestudy_path, ignore_errors=True)
results_data_path.mkdir(parents=True)

# Investment intervals and years between them
intervals = ["Interval_1", "Interval_2", "Interval_3"]
intervals_between_years = [10, 10]

# Emission reduction with respect to Interval 1
emission_reduction = {"Interval_2": 0.5, "Interval_3": 1.0}

## Load case study data
The data of the case study are provided with the package:
- `timeseries`: hourly electricity demand and import price, carbon price and subsidy, and climate data
- `carrier_constants`: all other carrier data (demands, import/export limits, prices and emission factors), which are
  constant over the year
- technology data: one JSON file per technology, specific for this case study

In [ ]:
case_study_data = adopt.load_multiyear_cluster_data()
timeseries = case_study_data["timeseries"]
carrier_constants = case_study_data["carrier_constants"]

tec_data_path = (
    Path(adopt.__file__).parent
    / "case_studies"
    / "data"
    / "multiyear_cluster"
    / "technology_data"
)

carrier_constants

## Create a case folder for each interval
Each interval is a separate optimization with its own input data folder (`Case_Interval_1`, `Case_Interval_2`, ...).
For each interval we:
- create the templates and adapt the topology: one node (NL), 21 carriers, and the interval as investment period
- adapt the model configuration: minimize costs in Interval 1 and costs at an emission limit afterwards, use 5
  typical days to reduce the computational time, and set a global discount rate of 5%
- create the input data folder, set the node location and define the technologies that can be built

In [ ]:
carriers = [
    "electricity",
    "hydrogen",
    "steam",
    "nitrogen",
    "CO2",
    "crackergas",
    "feedgas",
    "methane_bio",
    "methane",
    "syngas",
    "methanol",
    "olefins",
    "naphtha",
    "naphtha_bio",
    "ethylene",
    "propylene",
    "cracking_byproducts",
    "heat",
    "CO2captured",
    "ammonia",
    "HBfeed",
]

new_technologies = [
    "TestTec_Electrolyzer",
    "TestTec_SteamReformer",
    "TestTec_WGS_syngas",
    "TestTec_feedgas_mixer",
    "TestTec_BoilerEl",
    "TestTec_CO2emissions",
    "TestTec_eSMR_syngas",
    "TestTec_MeOHsynthesis",
    "TestTec_CrackerFurnace",
    "TestTec_eCrackerFurnace",
    "TestTec_OlefinSeparation",
    "TestTec_naphtha_mixer",
    "TestTec_SteamGenerator",
    "TestTec_CO2_mixer",
    "TestTec_HBfeed_mixer",
    "TestTec_HaberBosch",
]

for interval in intervals:
    interval_path = casestudy_path / f"Case_{interval}"
    interval_path.mkdir(parents=True, exist_ok=True)
    dp.create_optimization_templates(interval_path)

    # Topology
    with open(interval_path / "Topology.json", "r") as json_file:
        topology = json.load(json_file)
    topology["nodes"] = ["NL"]
    topology["carriers"] = carriers
    topology["investment_periods"] = [interval]
    with open(interval_path / "Topology.json", "w") as json_file:
        json.dump(topology, json_file, indent=4)

    # Model configuration
    with open(interval_path / "ConfigModel.json", "r") as json_file:
        configuration = json.load(json_file)
    if interval == intervals[0]:
        configuration["optimization"]["objective"]["value"] = "costs"
    else:
        configuration["optimization"]["objective"]["value"] = "costs_emissionlimit"
    configuration["optimization"]["typicaldays"]["N"]["value"] = 10
    configuration["optimization"]["typicaldays"]["method"]["value"] = 1
    configuration["economic"]["global_discountrate"]["value"] = 0.05
    configuration["solveroptions"]["mipgap"]["value"] = 0.01
    configuration["reporting"]["save_path"]["value"] = str(results_data_path)
    configuration["reporting"]["save_summary_path"]["value"] = str(results_data_path)
    with open(interval_path / "ConfigModel.json", "w") as json_file:
        json.dump(configuration, json_file, indent=4)

    # Input data folder
    dp.create_input_data_folder_template(interval_path)

    # Node location
    node_location = pd.read_csv(
        interval_path / "NodeLocations.csv", sep=";", index_col=0, header=0
    )
    node_location.at["NL", "lon"] = 4.90
    node_location.at["NL", "lat"] = 52.38
    node_location.at["NL", "alt"] = 10
    node_location = node_location.reset_index()
    node_location.to_csv(interval_path / "NodeLocations.csv", sep=";", index=False)

    # Technologies
    technologies_path = (
        interval_path / interval / "node_data" / "NL" / "Technologies.json"
    )
    with open(technologies_path, "r") as json_file:
        technologies = json.load(json_file)
    technologies["new"] = new_technologies
    with open(technologies_path, "w") as json_file:
        json.dump(technologies, json_file, indent=4)

    dp.copy_technology_data(interval_path, tec_data_path)

## Fill carrier data, carbon price and climate data
The input data are the same in all intervals: the constant carrier data and the hourly time series are written to the
input data folder of each interval.

In [ ]:
for interval in intervals:
    interval_path = casestudy_path / f"Case_{interval}"
    node_path = interval_path / interval / "node_data" / "NL"

    # Constant carrier data
    for _, row in carrier_constants.iterrows():
        dp.fill_carrier_data(
            interval_path,
            value_or_data=row["value"],
            columns=[row["column"]],
            carriers=[row["carrier"]],
            nodes=["NL"],
        )

    # Hourly electricity demand and import price
    dp.fill_carrier_data(
        interval_path,
        value_or_data=timeseries["electricity/Demand"],
        columns=["Demand"],
        carriers=["electricity"],
        nodes=["NL"],
    )
    dp.fill_carrier_data(
        interval_path,
        value_or_data=timeseries["electricity/Import price"],
        columns=["Import price"],
        carriers=["electricity"],
        nodes=["NL"],
    )

    # Carbon price and subsidy
    carbon_cost = pd.read_csv(
        node_path / "CarbonCost.csv", sep=";", index_col=0, header=0
    )
    carbon_cost["price"] = timeseries["CarbonCost/price"].values
    carbon_cost["subsidy"] = timeseries["CarbonCost/subsidy"].values
    carbon_cost = carbon_cost.reset_index()
    carbon_cost.to_csv(node_path / "CarbonCost.csv", sep=";", index=False)

    # Climate data
    climate_data = pd.read_csv(
        node_path / "ClimateData.csv", sep=";", index_col=0, header=0
    )
    climate_columns = [c for c in timeseries.columns if c.startswith("ClimateData/")]
    climate_data = pd.DataFrame(
        {c.replace("ClimateData/", ""): timeseries[c].values for c in climate_columns},
        index=climate_data.index,
    )
    climate_data = climate_data.reset_index()
    climate_data.to_csv(node_path / "ClimateData.csv", sep=";", index=False)

## Run the multiyear optimization
The intervals are solved one after the other. Before solving an interval, `installed_capacities_existing` takes the
capacities built in the previous interval and writes them as existing capacities into the input data of the current
interval. With `intervals_between_years`, the remaining lifetime of each carried-over capacity is checked: capacity
that reached the end of its lifetime is not carried over. The annualized capex of carried-over capacity is tracked for
the postprocessing.

The emission limit of Interval 2 and 3 is calculated from the net emissions of Interval 1.

In [ ]:
adopthub = {}
emissions = {}

for i, interval in enumerate(intervals):
    interval_path = casestudy_path / f"Case_{interval}"

    # Carry over the capacities of the previous interval
    if i != 0:
        prev_interval = intervals[i - 1]
        installed_capacities_existing(
            adopthub, interval, prev_interval, interval_path, intervals_between_years, i
        )
        del adopthub[prev_interval]

    adopthub[interval] = ModelHub()
    adopthub[interval].read_data(interval_path)

    # Add interval name as case name
    adopthub[interval].data.model_config["reporting"]["case_name"]["value"] = interval

    # Emission limit relative to Interval 1
    if interval in emission_reduction:
        adopthub[interval].data.model_config["optimization"]["emission_limit"][
            "value"
        ] = (1 - emission_reduction[interval]) * emissions[intervals[0]]

    adopthub[interval].quick_solve()

    model = adopthub[interval].model[
        adopthub[interval].info_solving_algorithms["aggregation_model"]
    ]
    emissions[interval] = model.var_emissions_net.value

## Postprocessing
The cost of each interval in the summary file only contains the capex of the capacity built in that interval. The
capacity carried over from previous intervals is existing capacity for the model, but its loan still has to be paid
until the end of its economic lifetime. The postprocessing adds this annualized capex to the summary file and
discounts the cost of each interval to the first interval.

```{note}
CCS that is retrofitted on an existing technology is carried over with its size, but its annualized capex is not
carried over to the next intervals. In this case study, CCS can be added to the existing cracker in Interval 2: the
cost of Interval 3 then does not include the capex of this CCS unit.
```

In [ ]:
summary_path = results_data_path / "Summary.xlsx"

add_values_to_summary(summary_path)
add_carry_over_annualization_to_summary(summary_path, casestudy_path, intervals)
add_discounted_cost_to_summary(
    summary_path, casestudy_path, intervals, intervals_between_years
)

## Results

In [ ]:
summary = pd.read_excel(summary_path)
summary[
    [
        "case",
        "emissions_net",
        "total_cost",
        "cost_annualization_carry_over_tecs",
        "total_cost_with_carry_over_annualization",
        "discount_factor",
        "discounted_total_cost_with_carry_over_annualization",
        "npv",
    ]
]

### Installed capacity
For each interval, the capacity of each technology is split into capacity carried over from previous intervals and
capacity newly built in the interval. In Interval 1 everything is new. In Interval 2, CCS is retrofitted on the existing
cracker to meet the emission limit. In Interval 3, electrified technologies (e-SMR, electrolyzer) are added to reach
zero net emissions. Sizes are in the unit of each technology, so they can be compared across intervals, not across
technologies.

<img src="figures/multiyear_capacity.png" width="700"/>

### Emissions and costs
Net emissions follow the limits: 50% of Interval 1 in Interval 2 and zero in Interval 3. The annual cost includes the
annualized capex of the capacity carried over from previous intervals. Export revenues are shown as negative costs.

<img src="figures/multiyear_emissions_cost.png" width="700"/>